In [1]:
"""
S01 baseline overflight — integrate notebooks 01–06.

Scenario:
  - One full polar overflight with seeded clouds over the target corridor (notebook 02)
  - 50-target meridian grid (notebook 01)
  - Nadir approach until lead margin before each target, then target engage
  - SequentialTargetBaselinePolicy requests [torque_nm, shutter_gym]; safety applies torque
  - Same training stack as MPO (training_episode_simulation_config; no OBC engage API)
  - Strided capture schedule: ep0 → 0, 5, 10, …, 45 (budget 10; warmup_capture_targets)
  - Skips intermediate corridor targets; long nadir coast between strided engages

Compare two attitude_request_mode arms (same policy + schedule):
  - torque: dim0 = RW torque request → AttitudeSafetyController
  - vector: dim0 = u → ObcPointingResolver (hold-last) → PD torque

Videos: artifacts/07-baseline-overflight-torque.mp4, 07-baseline-overflight-vector.mp4
Verification: s01_utils/baseline_overflight.py
"""

'\nS01 baseline overflight — integrate notebooks 01–06.\n\nScenario:\n  - One full polar overflight with seeded clouds over the target corridor (notebook 02)\n  - 50-target meridian grid (notebook 01)\n  - Nadir approach until lead margin before each target, then target engage\n  - SequentialTargetBaselinePolicy requests [torque_nm, shutter_gym]; safety applies torque\n  - Same training stack as MPO (training_episode_simulation_config; no OBC engage API)\n  - Strided capture schedule: ep0 → 0, 5, 10, …, 45 (budget 10; warmup_capture_targets)\n  - Skips intermediate corridor targets; long nadir coast between strided engages\n\nCompare two attitude_request_mode arms (same policy + schedule):\n  - torque: dim0 = RW torque request → AttitudeSafetyController\n  - vector: dim0 = u → ObcPointingResolver (hold-last) → PD torque\n\nVideos: artifacts/07-baseline-overflight-torque.mp4, 07-baseline-overflight-vector.mp4\nVerification: s01_utils/baseline_overflight.py\n'

In [2]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
backend_root = notebook_dir
for _ in range(6):
    if (backend_root / "simulation").is_dir():
        break
    backend_root = backend_root.parent
os.chdir(backend_root)
sys.path.insert(0, str(backend_root))
_s01_dir = backend_root / "notebooks" / "s01"
sys.path.insert(0, str(_s01_dir))
print(f"backend_root={backend_root}")

backend_root=d:\code\sem-proj-asc\backend


In [3]:
import importlib

import s01_utils.baseline_overflight as bof

importlib.reload(bof)

setup = bof.build_baseline_overflight_setup()
bof.print_baseline_setup_summary(setup)

Baseline overflight setup
  targets:           50
  clouds:            27 (seeded over target corridor)
  lead margin:       20.0° before target engage
  altitude:          548.2 km
  orbit window:      None° .. None° (auto if unset)
  capture budget:    10/orbit


In [4]:
rollout_torque = bof.run_baseline_overflight_rollout(
    setup, show_progress=True, attitude_request_mode="torque"
)
rollout_vector = bof.run_baseline_overflight_rollout(
    setup, show_progress=True, attitude_request_mode="vector"
)
for label, rollout in [("torque", rollout_torque), ("vector", rollout_vector)]:
    print(
        f"{label}: steps={rollout.series.t_s.shape[0]}  "
        f"shutter_cmds={len(rollout.cmd_steps)}  "
        f"safety_events={rollout.n_safety_events}"
    )

d:\code\sem-proj-asc\backend\simulation\stepper.py:177: UserWarning: controller_update_interval (1 s) is not an integer multiple of simulation_timestep (0.4 s); using nearest multiple: 0.8 s (2 sim steps).
  self._controller_interval_steps, self._effective_controller_interval_s = resolve_controller_interval_steps(
baseline overflight (vector): 100%|██████████| 1965/1965 [00:10<00:00, 191.41it/s]

torque: steps=1966  shutter_cmds=10  safety_events=0
vector: steps=1966  shutter_cmds=10  safety_events=0


In [5]:
kpis_torque = bof.evaluate_baseline_capture_results(
    rollout_torque.series, rollout_torque.cmd_steps
)
kpis_vector = bof.evaluate_baseline_capture_results(
    rollout_vector.series, rollout_vector.cmd_steps
)

bof.print_baseline_mode_comparison(
    torque=(rollout_torque, kpis_torque),
    vector=(rollout_vector, kpis_vector),
)
print()
bof.print_baseline_capture_kpis(
    kpis_torque, rollout=rollout_torque, title="Torque mode — capture KPIs"
)
print()
bof.print_baseline_capture_kpis(
    kpis_vector, rollout=rollout_vector, title="Vector mode — capture KPIs"
)

Baseline torque vs vector comparison
  metric                    torque        vector
  attitude mode             torque        vector
  capture schedule    [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]  [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]
  shutter cmds                  10            10
  captures taken                10            10
  applied capture           415.26        443.21
  mean quality              0.9095        0.9225
  safety events                  0             0
  mean sim reward            0.000         0.000
  cmd_steps match              yes           yes

Torque mode — capture KPIs
  shutter commands:  10 / 50 targets
  captures taken:    10 / 10 budget
  latent capture:    415.26  (k * cov * quality * (1 - cloud))
  applied capture:   415.26  (latent scaled by cov; 0 if not visible / not taken / repeat target)
  mean quality:      0.9095
  attitude safety events: 0
  mean sim reward:   0.000

  cmd  cap  taken  visible  cov    quality  cloud   latent  applied  budget

In [6]:
import matplotlib

matplotlib.use("Agg")

ARTIFACT_DIR = backend_root / "notebooks" / "s01" / "artifacts"

reward_trace_torque = bof.build_baseline_latent_reward_trace(
    rollout_torque.series, kpis_torque.capture_results
)
reward_trace_vector = bof.build_baseline_latent_reward_trace(
    rollout_vector.series, kpis_vector.capture_results
)

video_torque = bof.export_baseline_overflight_video(
    rollout_torque.series,
    reward_trace_torque,
    ARTIFACT_DIR / "07-baseline-overflight-torque.mp4",
    shutter_cmd_steps=rollout_torque.cmd_steps,
)
video_vector = bof.export_baseline_overflight_video(
    rollout_vector.series,
    reward_trace_vector,
    ARTIFACT_DIR / "07-baseline-overflight-vector.mp4",
    shutter_cmd_steps=rollout_vector.cmd_steps,
)
print(f"torque={video_torque}")
print(f"vector={video_vector}")

[video] archived previous export -> D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\video_archive\002-07-baseline-overflight-torque.mp4
[video] parallel export workers=8 frames=438 encoder=h264_nvenc via=imageio_ffmpeg fps=20
[mpo_video:after_export] 07-baseline-overflight-torque.mp4 (2784122 bytes, codec=h264)
[mpo_video:play] 07-baseline-overflight-torque.mp4 (2784122 bytes, codec=h264)


artifact=D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\07-baseline-overflight-torque.mp4
[video] archived previous export -> D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\video_archive\002-07-baseline-overflight-vector.mp4
[video] parallel export workers=8 frames=438 encoder=h264_nvenc via=imageio_ffmpeg fps=20
[mpo_video:after_export] 07-baseline-overflight-vector.mp4 (2810794 bytes, codec=h264)
[mpo_video:play] 07-baseline-overflight-vector.mp4 (2810794 bytes, codec=h264)


artifact=D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\07-baseline-overflight-vector.mp4
torque=D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\07-baseline-overflight-torque.mp4
vector=D:\code\sem-proj-asc\backend\notebooks\s01\artifacts\07-baseline-overflight-vector.mp4
